In [ ]:
# import requests
# import xml.etree.ElementTree as ET
# from typing import List, Dict

# BASE_URL = "https://www.bizinfo.go.kr/uss/rss/bizinfoApi.do"

# def fetch_tech_notices(api_key: str, count: int = 20) -> List[Dict[str, str]]:
#     params = {
#         "crtfcKey": api_key,
#         "dataType": "rss",
#         "pageIndex": 1,
#         "pageUnit": count,  # 여러 개를 가져와서 필터링
#         "searchCnt": count,
#     }
    
#     r = requests.get(BASE_URL, params=params, timeout=20)
#     r.raise_for_status()

#     root = ET.fromstring(r.content)
#     items = root.findall(".//item")
    
#     tech_notices = []

#     for item in items:
#         # lcategory 또는 pldirSportRealmLclasCodeNm 태그 확인
#         lcat = item.find("lcategory")
#         pldir = item.find("pldirSportRealmLclasCodeNm")
        
#         # 두 태그 중 하나라도 '기술'을 포함하고 있는지 확인
#         category_text = ""
#         if lcat is not None and lcat.text: category_text = lcat.text
#         elif pldir is not None and pldir.text: category_text = pldir.text

#         if "기술" in category_text:
#             # 해당 공고의 모든 컬럼 추출
#             notice_data = {}
#             for child in item:
#                 notice_data[child.tag] = child.text.strip() if child.text else "N/A"
#             tech_notices.append(notice_data)

#     return tech_notices

# # ===== 실행 및 결과 출력 =====
# API_KEY = "O5T1ww" 
# try:
#     # 최신 공고 50개 중 '기술' 분야만 추출
#     results = fetch_tech_notices(API_KEY, count=50)
    
#     print(f"--- '기술' 분야 공고 검색 결과: {len(results)}건 ---")
#     print("-" * 60)
    
#     for idx, notice in enumerate(results, 1):
#         print(f"[{idx}] 제목: {notice.get('title') or notice.get('pblancNm')}")
#         print(f"    분야: {notice.get('lcategory') or notice.get('pldirSportRealmLclasCodeNm')}")
#         print(f"    기관: {notice.get('author') or notice.get('jrsdInsttNm')}")
#         print(f"    URL: {notice.get('link') or notice.get('pblancUrl')}")
#         print("-" * 60)

#     if not results:
#         print("최근 공고 중 '기술' 분야의 공고가 없습니다. count 범위를 늘려보세요.")

# except Exception as e:
#     print(f"오류: {e}")

--- '기술' 분야 공고 검색 결과: 8건 ---
------------------------------------------------------------
[1] 제목: [인천] 2026년 가상융합산업 콘텐츠 제작 장비 임차지원 공고(인천 가상융합산업 혁신센터)
    분야: 기술
    기관: 인천광역시
    URL: https://www.bizinfo.go.kr/web/lay1/bbs/S1T122C128/AS/74/view.do?pblancId=PBLN_000000000117307
------------------------------------------------------------
[2] 제목: 2026년 한국식품산업클러스터진흥원 기능성표시식품개발 기술지원사업 모집 공고
    분야: 기술
    기관: 농림축산식품부
    URL: https://www.bizinfo.go.kr/web/lay1/bbs/S1T122C128/AS/74/view.do?pblancId=PBLN_000000000117288
------------------------------------------------------------
[3] 제목: [인천] 2026년 글로벌 IP스타기업 육성(IP기반 해외진출지원) 모집 공고
    분야: 기술
    기관: 인천광역시
    URL: https://www.bizinfo.go.kr/web/lay1/bbs/S1T122C128/AS/74/view.do?pblancId=PBLN_000000000117272
------------------------------------------------------------
[4] 제목: [경남] 진주시 2026년 GAP 안전성 분석(안전성 검사비) 지원사업 신청 공고
    분야: 기술
    기관: 경상남도
    URL: https://www.bizinfo.go.kr/web/lay1/bbs/S1T122C128/AS/74/view.do?pblancId=PBLN_0000000001172

In [4]:
pip install pymysql

Note: you may need to restart the kernel to use updated packages.


In [14]:
import requests
import xml.etree.ElementTree as ET
import pymysql
# 설정 파일에서 정보를 가져옵니다.
from config import DB_CONFIG, API_KEY, BASE_URL 

def fetch_and_save():
    # 1. API 호출
    params = {
        "crtfcKey": API_KEY,
        "dataType": "rss",
        "pageIndex": 1,
        "pageUnit": 300,
        "searchCnt": 500
    }
    
    try:
        r = requests.get(BASE_URL, params=params, timeout=20)
        r.raise_for_status()
        root = ET.fromstring(r.content)
        items = root.findall(".//item")
        print(f"API 호출 성공: 총 {len(items)}건 확인")
    except Exception as e:
        print(f"API 호출 실패: {e}")
        return

    # 2. DB 연결
    conn = None
    try:
        conn = pymysql.connect(**DB_CONFIG)
        print("bnb DB 연결 성공")
        
        with conn.cursor() as cursor:
            new_count = 0
            
            for item in items:
                # 모든 태그를 먼저 추출
                notice = {}
                for child in item:
                    notice[child.tag] = child.text.strip() if child.text else ""

                # 필터링 조건
                lcat = notice.get("lcategory", "")
                pldir = notice.get("pldirSportRealmLclasCodeNm", "")
                title = notice.get("pblancNm", "")

                if "기술" in lcat or "기술" in pldir or "기술" in title:
                    # 중복 체크용 고유 ID (pblancId)
                    seq = notice.get("pblancId")
                    if not seq: continue

                    cursor.execute("SELECT 1 FROM project_notices WHERE seq = %s", (seq,))
                    if cursor.fetchone():
                        continue
                    
                    # 3. DB 적재
                    sql = """
                    INSERT INTO project_notices 
                    (title, link, seq, author, excInsttNm, description, pubDate, reqstDt, trgetNm, printFlpthNm, printFileNm, hashTags) 
                    VALUES (%s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s)
                    """
                    
                    # 성공한 데이터에서 각 값들을 맵핑
                    cursor.execute(sql, (
                        notice.get("pblancNm"),           # title
                        notice.get("pblancUrl"),          # link
                        seq,                              # seq
                        notice.get("jrsdInsttNm"),        # author (소관기관)
                        notice.get("excInsttNm"),         # 수행기관
                        notice.get("bsnsSumryCn"),        # description (사업개요)
                        notice.get("creatPnttm"),         # pubDate (등록일자)
                        notice.get("reqstBeginEndDe"),    # reqstDt (신청기간)
                        notice.get("trgetNm"),            # trgetNm (지원대상)
                        notice.get("printFlpthNm"),       # 본문출력파일경로
                        notice.get("printFileNm"),        # 본문출력파일명
                        notice.get("hashtags")            # 해시태그
                    ))
                    new_count += 1
            
            conn.commit()
            print(f"결과: 새로운 기술 공고 {new_count}건")
            
    except pymysql.MySQLError as e:
        print(f"DB 에러: {e}")
    finally:
        if conn:
            conn.close()
            print("DB 연결 종료")

if __name__ == "__main__":
    fetch_and_save()

API 호출 성공: 총 300건 확인
bnb DB 연결 성공
결과: 새로운 기술 공고 32건
DB 연결 종료
